# Khảo sát MixLinear — C0: chỉ nhánh phụ, KHÔNG có MixLinear

## Cấu hình của notebook này — C0, 929 tham số

```
(batch, 200) → trừ trung bình
             → Linear(200, 4)  →  Linear(4, 25)      không có hàm kích hoạt
             → cộng lại trung bình → (batch, 25)
```

**Không có MixLinear.** Đây là đối chứng: nếu nhánh phụ đứng một mình cũng đạt
điểm ngang khi ghép vào MixLinear (C2), thì chưa thấy phần MixLinear đóng góp
gì trong điều kiện đã thử.

Xử lý trung bình y hệt MixLinear — trừ trước, cộng lại sau. Không có bước này
thì đối chứng khác cả cách tiền xử lý, không so được nữa.

**Không phải `Linear(200, 25)` đầy đủ:**

| | tham số |
|---|---:|
| `Linear(200, 25)` đầy đủ | 5.025 |
| tách làm hai lớp qua chiều 4 | **929** |

Hai lớp không phi tuyến hợp lại vẫn là **một** phép affine, chỉ khác là ma trận
bị ép hạng tối đa 4. Đây là cách tiết kiệm tham số, không phải cách tăng khả
năng biểu diễn.

## Vì sao có khảo sát này

MixLinear-63 đã chạy: `cv_mean 0,672429 ± 0,006570`. Đường hội tụ cho thấy nó
**đã học hết** — epoch 18→19 chỉ còn giảm 0,28%, phẳng hơn cả TCN-64 — nhưng
dừng ở `train_mse 0,0500`, gấp 3,8 lần TCN. Là giới hạn **sức chứa**, không
phải thiếu epoch.

Mà MixLinear gần kịch trần thiết kế của chính nó: hai lớp `FLinear` nối nhau
không có phi tuyến nên hợp lại chỉ là một ma trận, hạng tối đa 3, đạt được từ
79 tham số. Thêm tham số vào nhánh cũ **không thể** thêm khả năng biểu diễn.
Muốn có sức chứa thật thì phải thêm đường đi mới.

## Thiết kế giai thừa — ba notebook chạy song song

|  | không nhánh phụ | có nhánh phụ |
|---|---|---|
| **không MixLinear** | — | **C0** (929) |
| **có MixLinear** | **B0** (63, đã chạy) | **C2** (992) |

Thêm **C3** (992): giống hệt C2, chỉ khác một `GELU`.

| notebook | model | tham số |
|---|---|---:|
| `TN_MixLinear_C0.ipynb` | `low_rank_linear` | 929 |
| `TN_MixLinear_C2.ipynb` | `mix_linear_linear` | 992 |
| `TN_MixLinear_C3.ipynb` | `mix_linear_mlp` | 992 |

Ba notebook **độc lập hoàn toàn**, mỗi cái nén ra tên zip riêng nên chạy cùng
lúc ở ba phiên Colab không đè nhau. Mỗi cái khoảng **45 phút**.

Bốn phép so đọc được sau khi cả ba xong:

| so sánh | trả lời câu gì | sạch không |
|---|---|---|
| **C3 − C2** | phi tuyến giúp gì | **sạch** — cùng 992 tham số, khác đúng GELU |
| **C2 − C0** | thêm MixLinear khi đã có nhánh phụ | lệch 63 tham số |
| **C2 − B0** | thêm nhánh phụ khi đã có MixLinear | lệch 929 tham số |
| C0 − B0 | 929 tham số tuyến tính so với 63 tham số MixLinear | đổi cả hai biến |

## Ba điều phải ghi khi báo cáo

**1. Nhánh phụ là đề xuất của đồ án, không phải MixLinear nguyên bản.** LSTNet
(SIGIR 2018) và N-BEATS (ICLR 2020) là tiền lệ cho việc ghép thành phần tuyến
tính với phi tuyến, **không** phải bằng chứng cho cấu hình chiều 4 trên UWB.

**2. C2 và C3 không bằng nhau tuyệt đối.** Ở C2, `bias` lớp đầu bị hấp thụ vào
`bias` lớp sau, vì hai `Linear` không phi tuyến hợp lại thành đúng một phép
affine. C2 có **988** tham số tác dụng độc lập so với 992 của C3. Chênh 0,4%,
không hỏng phép so, nhưng đừng viết "cùng hệt số tham số".

**3. Một seed là sàng lọc, không phải kiểm định.** B0 có `seed_std` **0,0066**,
ba seed trải 0,6648–0,6764. Chênh lệch dưới khoảng **0,007** trên một seed là
ngẫu nhiên. So với **`0,6760` của B0 seed 0**, không so với trung bình ba seed.

Và vì train chung từ đầu, **không** được nói nhánh phụ "học phần sai của
MixLinear" — nó chỉ là đường đi song song, hai bên học cùng nhau.

Thiết kế đầy đủ: `docs/THIET_KE_MIXLINEAR_DE_CLAUDE_REVIEW.md`

## 1. Chuẩn bị Colab

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Tải mã nguồn rồi vào thư mục đó.

In [2]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

/content/UWB_RADAR

thư mục làm việc : /content/UWB_RADAR
commit đồ án     : 5c95b56
commit MobiVital : 4319731 (đã ghim)
GPU              : Tesla T4, 15360 MiB


Lấy `by_user/` và `windows/` từ Drive.

In [3]:
!python scripts/restore_processed_data_on_drive.py

by_user   : bung /content/drive/MyDrive/mobivital/by_user.tar ...
            12 tệp
windows   : bung /content/drive/MyDrive/mobivital/windows.tar.gz ...
            dev_cv 8 tệp, final_train có

2.5G	data/processed/by_user
503M	data/processed/windows


## 2. Kiểm bản cài đặt

Sáu phép kiểm. Mục 6 xác nhận đúng điều nói ở trên: hai lớp không phi tuyến hợp
lại chỉ là một ma trận hạng tối đa 4, và in kèm số tham số của `Linear(200, 25)`
đầy đủ để thấy phép tách tiết kiệm bao nhiêu.

In [4]:
!python scripts/check_model.py --model low_rank_linear

Kiểm model: low_rank_linear

1. Hình dạng vào ra và giá trị
   vào (4, 200)  ->  ra (4, 25)
   ra đúng (4, 25)                                            đạt
   mọi giá trị hữu hạn                                        đạt

2. Gradient
   4/4 tham số nhận được gradient
   mọi tham số đều lan ngược tới                              đạt

3. Số tham số
   929

4. Lưu và nạp lại state_dict
   nạp lại cho ra đúng đầu ra cũ                              đạt

5. Riêng LowRankLinear
   đúng hai lớp Linear                                        đạt
   lớp đầu ép 200 chiều xuống 4                               đạt
   cả hai lớp đều có bias                                     đạt

6. Hai lớp không phi tuyến = MỘT phép affine hạng <= 4
   tích hai ma trận (25, 200), hạng 4
   hạng không vượt 4                                          đạt
   Linear(200, 25) đầy đủ sẽ tốn 5025 tham số, ở đây 929

TẤT CẢ ĐẠT — bản cài đặt dùng được.


## 3. Chạy 4 fold, MỘT seed

Giao thức giữ nguyên: 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9, bốn
fold cũ. Chỉ đổi kiến trúc.

Tên cấu hình là `low_rank_linear_h4_mse_corr0.9_seed0`, ghi vào
`runs/tn_mixlinear_ablation/`. Khoảng **45 phút** — 4,6 phút mỗi fold train
cộng khoảng 25 phút chấm điểm.

In [5]:
!python scripts/run_cv.py --experiment tn_mixlinear_ablation --model low_rank_linear --seed 0

thực nghiệm tn_mixlinear_ablation  -> runs/tn_mixlinear_ablation/
cấu hình low_rank_linear_h4_mse_corr0.9_seed0
thiết bị  Tesla T4

----------------------------------------------------------
val_AB  train CDEFKL  chấm AB
210964 cửa sổ train
epoch  0  mse 0.15767  pearson 0.0238   0.1 phút
epoch  1  mse 0.04945  pearson 0.2388   0.2 phút
epoch  2  mse 0.03875  pearson 0.4099   0.4 phút
epoch  3  mse 0.03555  pearson 0.4389   0.5 phút
epoch  4  mse 0.03410  pearson 0.4445   0.6 phút
epoch  5  mse 0.03325  pearson 0.4465   0.7 phút
epoch  6  mse 0.03264  pearson 0.4480   0.8 phút
epoch  7  mse 0.03218  pearson 0.4490   1.0 phút
epoch  8  mse 0.03182  pearson 0.4500   1.1 phút
epoch  9  mse 0.03155  pearson 0.4506   1.2 phút
epoch 10  mse 0.03130  pearson 0.4514   1.3 phút
epoch 11  mse 0.03109  pearson 0.4517   1.4 phút
epoch 12  mse 0.03094  pearson 0.4526   1.6 phút
epoch 13  mse 0.03078  pearson 0.4530   1.7 phút
epoch 14  mse 0.03066  pearson 0.4532   1.8 phút
epoch 15  mse 0.03055  p

## 4. Cất kết quả

Nén ra tên riêng `tn_mixlinear_c0.zip`, khác hai notebook kia, nên chạy song song
không đè nhau trên Drive.

In [6]:
!python scripts/save_results.py tn_mixlinear_ablation --out tn_mixlinear_c0

runs/tn_mixlinear_ablation/  ->  runs/tn_mixlinear_c0.zip   (0.0 MB)
   5 dòng metric trong summary.csv

Bên trong:
        0  2026-09-07 09:52   tn_mixlinear_ablation/
        0  2026-09-07 09:29   tn_mixlinear_ablation/low_rank_linear_h4_mse_corr0.9_seed0_val_AB/
        0  2026-09-07 09:36   tn_mixlinear_ablation/low_rank_linear_h4_mse_corr0.9_seed0_val_CE/
        0  2026-09-07 09:43   tn_mixlinear_ablation/low_rank_linear_h4_mse_corr0.9_seed0_val_DF/
        0  2026-09-07 09:49   tn_mixlinear_ablation/low_rank_linear_h4_mse_corr0.9_seed0_val_KL/
    16927  2026-09-07 09:52   tn_mixlinear_ablation/scores_low_rank_linear_h4_mse_corr0.9_seed0_val_KL.csv
      189  2026-09-07 09:53   tn_mixlinear_ablation/README.txt
    21741  2026-09-07 09:40   tn_mixlinear_ablation/scores_low_rank_linear_h4_mse_corr0.9_seed0_val_CE.csv
    24339  2026-09-07 09:33   tn_mixlinear_ablation/scores_low_rank_linear_h4_mse_corr0.9_seed0_val_AB.csv
    19948  2026-09-07 09:47   tn_mixlinear_ablation/scores_

## 5. Đường hội tụ

Cùng câu hỏi đã hỏi cho B0: điểm thấp là do hết sức chứa hay do 20 epoch chưa
đủ? B0 dừng ở `train_mse` **0,0500** và đã phẳng. Cấu hình này nhiều tham số
hơn nên đáng lẽ phải xuống thấp hơn — nếu không thì phần sức chứa thêm vào
không được dùng.

In [7]:
!cut -d, -f1,2 runs/tn_mixlinear_ablation/low_rank_linear_h4_mse_corr0.9_seed0_val_AB/curve.csv | tail -6

14,0.030655160166090423
15,0.030550418125873974
16,0.03044698583676819
17,0.030352336183090915
18,0.03026862085754804
19,0.030190269079727137


## 6. Bảng so

Chỉ thấy cấu hình của phiên này. Mốc để đặt cạnh:

| | tham số | cv_mean |
|---|---:|---:|
| **B0 MixLinear, seed 0** | 63 | **0,6760** |
| B0 MixLinear, 3 seed | 63 | 0,6724 ± 0,0066 |
| LSTM-67 | 56.908 | 0,7532 |

So **một seed với một seed**: dùng `0,6760`, không dùng trung bình ba seed. Và
nhớ ngưỡng nhiễu **0,007**.

Bảng gộp cả ba cấu hình dựng sau, khi đủ ba tệp zip trên Drive.

In [8]:
!python scripts/compare_cv.py --experiment tn_mixlinear_ablation


BẢNG 1 — cv_score, thực nghiệm tn_mixlinear_ablation
cấu hình                         tham số  seed    cv_mean  seed_std  fold_std   từng seed
--------------------------------------------------------------------------------------------------------------
low_rank_linear_h4_mse_corr0.9       929     1   0.654737       N/A  0.109602   s0 0.6547

cv_mean  = trung bình cv_score của các seed. cv_score của một seed là
           trung bình điểm macro 4 fold; macro = trung bình theo NGƯỜI.
seed_std = dao động giữa các seed. Chênh lệch giữa hai cấu hình nhỏ hơn
           số này thì chưa kết luận được.
fold_std = dao động giữa 4 fold, trung bình trên các seed. Nói dữ liệu
           giữa các người khác nhau ra sao, KHÔNG dùng để so cấu hình.



## 7. Ngắt phiên

Kết quả đã nén sang Drive ở mục 4 nên ngắt ở đây không mất gì.

In [ ]:
from google.colab import runtime
runtime.unassign()